# Oral Cavity Segmentation - DeepLabV3+ (ResNet34 Encoder + ASPP)

This notebook trains **only the DeepLabV3+ (ResNet34 Encoder + ASPP) model** on the oral-cavity masks (256 x 256, batch 8, 25 epochs, Adam 1e-4, BCE + Dice loss), so it stays light on RAM/disk compared to running all three models in one session.

The other two segmentation models have their own notebooks, and the **Model Comparison** notebook combines the Dice/IoU curves from all three.

> **Note:** this is the training code for this model. Cell outputs are not stored in this copy; the training history, curves and summary from the original Kaggle run (August 2026) are in `results/segmentation/deeplabv3plus_results/`, and the three models are compared in `comparison.ipynb`.

## 1. Environment Check

In [ ]:
import sys, platform
import tensorflow as tf

print("Python:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))


## Kaggle Input Check

In [ ]:
import os
print("Available Kaggle input datasets:")
for d in os.listdir("/kaggle/input"):
    print(" -", d)


## Imports

In [ ]:
import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## Config & Output Folder

In [ ]:
MODEL_NAME = "deeplabv3plus"

OUTPUT_DIR = Path("/kaggle/working/Segmentation Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASKS_DIR = OUTPUT_DIR / "masks"
MASKS_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = OUTPUT_DIR / f"{MODEL_NAME}_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SEGMENTED_DIR = RESULT_DIR / "segmented_samples"
SEGMENTED_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 25

print("Output folder:", OUTPUT_DIR)
print("This model's result folder:", RESULT_DIR)


## 2. Dataset `Categories` Folder

In [ ]:
DATASET_DIR = Path(
    "/kaggle/input/datasets/annapoornaak/oral-cancer-dataset-mtech-project"
)

print("Categories folder:", DATASET_DIR)
print("Exists:", DATASET_DIR.exists())


## 3. COCO Annotation JSON

In [ ]:
ANNOTATION_FILE = Path(
    "/kaggle/input/datasets/annapoornaak/metadata-zenodo/Annotation.json"
)

print("Annotation file:", ANNOTATION_FILE)
print("Exists:", ANNOTATION_FILE.exists())


## 4. Load & Verify `Lesion` and `Oral Cavity` Categories

In [ ]:
with open(ANNOTATION_FILE, "r") as f:
    coco = json.load(f)

print("Images:", len(coco["images"]))
print("Annotations:", len(coco["annotations"]))
print("Categories:", coco["categories"])

name_to_id = {c["name"]: c["id"] for c in coco["categories"]}
assert "Lesion" in name_to_id, "Lesion category not found"
assert "Oral Cavity" in name_to_id, "Oral Cavity category not found"

CATEGORY_ID_LESION = name_to_id["Lesion"]
CATEGORY_ID_ORAL = name_to_id["Oral Cavity"]

print("Lesion id:", CATEGORY_ID_LESION)
print("Oral Cavity id:", CATEGORY_ID_ORAL)

category_counts = Counter(ann["category_id"] for ann in coco["annotations"])
for c in coco["categories"]:
    print(c["name"], "->", category_counts[c["id"]], "annotations")


## 5. Match Annotation Images to Category Folders

In [ ]:
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

filename_to_path = {}
for path in DATASET_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        filename_to_path[path.name] = path

images_info = {img["id"]: img for img in coco["images"]}

matched, unmatched = 0, 0
image_id_to_path = {}

for image_id, info in images_info.items():
    fname = Path(info["file_name"]).name
    if fname in filename_to_path:
        image_id_to_path[image_id] = filename_to_path[fname]
        matched += 1
    else:
        unmatched += 1

print("Matched:", matched)
print("Unmatched:", unmatched)


## 6. Create Oral Cavity Binary Masks

In [ ]:
oral_annotations_by_image = {}
for ann in coco["annotations"]:
    if ann["category_id"] != CATEGORY_ID_ORAL:
        continue
    oral_annotations_by_image.setdefault(ann["image_id"], []).append(ann)

print("Images with oral cavity annotations:", len(oral_annotations_by_image))

def create_oral_cavity_mask(image_id):
    image_path = image_id_to_path.get(image_id)
    if image_path is None:
        return None

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    height, width = image.shape[:2]
    # Black binary mask
    mask = np.zeros((height, width), dtype=np.uint8)

    for ann in oral_annotations_by_image.get(image_id, []):
        segmentation = ann.get("segmentation", [])
        # Skip invalid segmentation
        if not segmentation:
            continue

        # COCO polygon can be either:
        # [x1,y1,x2,y2,...]
        # OR
        # [[x1,y1,x2,y2,...], [...]]

        if isinstance(segmentation[0], (int, float)):
            polygons = [segmentation]
        else:
            polygons = segmentation

        for polygon in polygons:

            # Make sure polygon is valid
            if len(polygon) < 6:
                continue

            points = np.array(
                polygon,
                dtype=np.float32
            ).reshape(-1, 2)

            points = np.round(points).astype(np.int32)

            # Keep coordinates inside image
            points[:, 0] = np.clip(
                points[:, 0],
                0,
                width - 1
            )

            points[:, 1] = np.clip(
                points[:, 1],
                0,
                height - 1
            )

            # Fill polygon
            cv2.fillPoly(
                mask,
                [points],
                255
            )

    return mask

test_image_id = list(oral_annotations_by_image.keys())[0]
test_mask = create_oral_cavity_mask(test_image_id)
print("Test image ID:", test_image_id)
print("Mask shape:", test_mask.shape)
print("Unique values:", np.unique(test_mask))


## 7. Generate Complete Image-Mask Dataset

In [ ]:
from tqdm import tqdm
records = []

for image_id in tqdm(list(oral_annotations_by_image.keys())):
    image_path = image_id_to_path.get(image_id)
    if image_path is None:
        continue

    mask = create_oral_cavity_mask(image_id)
    if mask is None:
        continue

    mask_path = MASKS_DIR / f"{image_id}.png"
    cv2.imwrite(str(mask_path), mask)

    records.append({
        "image_id": image_id,
        "image_path": str(image_path),
        "mask_path": str(mask_path),
    })

dataset_df = pd.DataFrame(records)
dataset_df.to_csv(OUTPUT_DIR / "dataset_index.csv", index=False)

print("Total image-mask pairs:", len(dataset_df))
dataset_df.head()


## 8. 80/20 Train-Validation Split

In [ ]:
train_df, val_df = train_test_split(dataset_df, test_size=0.2, random_state=SEED)
print("Train:", len(train_df))
print("Val:", len(val_df))


## 9. Paired Image + Mask Augmentation Pipeline

In [ ]:
def load_image_mask(image_path, mask_path):
    image_path = image_path.numpy().decode("utf-8")
    mask_path = mask_path.numpy().decode("utf-8")

    image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    image = image.astype(np.float32) / 255.0

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Could not read mask: {mask_path}")
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    return image, mask

def tf_load_image_mask(image_path, mask_path):
    image, mask = tf.py_function(
        func=load_image_mask,
        inp=[image_path, mask_path],
        Tout=[tf.float32, tf.float32]
    )
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return image, mask

def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, mask

def make_dataset(df, training):
    paths = (df["image_path"].values, df["mask_path"].values)
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(tf_load_image_mask, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(256)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

for img_batch, mask_batch in train_ds.take(1):
    print("Image batch:", img_batch.shape)
    print("Mask batch:", mask_batch.shape)


## Loss & Metrics

In [ ]:
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


## ResNet34-style Encoder (shared backbone)

In [ ]:
def conv_bn_relu(x, filters, kernel=3, strides=1):
    x = layers.Conv2D(filters, kernel, strides=strides, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def basic_block(x, filters, strides=1):
    shortcut = x
    y = conv_bn_relu(x, filters, 3, strides)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if strides != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=strides, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    y = layers.Add()([y, shortcut])
    return layers.Activation("relu")(y)

def resnet34_encoder(inputs):
    x = conv_bn_relu(inputs, 64, 7, 2)
    skip1 = x
    x = layers.MaxPooling2D(3, 2, padding="same")(x)

    for i in range(3):
        x = basic_block(x, 64, strides=1)
    skip2 = x

    for i, s in enumerate([2, 1, 1, 1]):
        x = basic_block(x, 128, strides=s)
    skip3 = x

    for i, s in enumerate([2, 1, 1, 1, 1, 1]):
        x = basic_block(x, 256, strides=s)
    skip4 = x

    for i, s in enumerate([2, 1, 1]):
        x = basic_block(x, 512, strides=s)

    return x, [skip4, skip3, skip2, skip1]


## DeepLabV3+ (ASPP-based)

In [ ]:
def aspp_block(x, filters=256):
    dims = x.shape

    conv_1x1 = conv_bn_relu(x, filters, 1)

    conv_r6 = layers.Conv2D(filters, 3, padding="same", dilation_rate=6, use_bias=False)(x)
    conv_r6 = layers.BatchNormalization()(conv_r6)
    conv_r6 = layers.Activation("relu")(conv_r6)

    conv_r12 = layers.Conv2D(filters, 3, padding="same", dilation_rate=12, use_bias=False)(x)
    conv_r12 = layers.BatchNormalization()(conv_r12)
    conv_r12 = layers.Activation("relu")(conv_r12)

    conv_r18 = layers.Conv2D(filters, 3, padding="same", dilation_rate=18, use_bias=False)(x)
    conv_r18 = layers.BatchNormalization()(conv_r18)
    conv_r18 = layers.Activation("relu")(conv_r18)

    pool = layers.GlobalAveragePooling2D(keepdims=True)(x)
    pool = conv_bn_relu(pool, filters, 1)
    pool = layers.Resizing(dims[1], dims[2], interpolation="bilinear")(pool)

    x = layers.Concatenate()([conv_1x1, conv_r6, conv_r12, conv_r18, pool])
    x = conv_bn_relu(x, filters, 1)
    return x

def build_deeplabv3plus(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = layers.Input(shape=input_shape)
    bottleneck, skips = resnet34_encoder(inputs)
    low_level = skips[2]  # 1/4 resolution, 64 channels

    x = aspp_block(bottleneck, 256)
    x = layers.Resizing(low_level.shape[1], low_level.shape[2], interpolation="bilinear")(x)

    low_level = conv_bn_relu(low_level, 48, 1)
    x = layers.Concatenate()([x, low_level])

    x = conv_bn_relu(x, 256)
    x = conv_bn_relu(x, 256)

    x = layers.Resizing(IMG_SIZE, IMG_SIZE, interpolation="bilinear")(x)
    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)

    return models.Model(inputs, outputs, name="deeplabv3plus")


## Sanity Check: Corrupted Images

In [ ]:
from PIL import Image

bad_images = []

for idx, row in dataset_df.iterrows():
    image_path = row["image_path"]

    try:
        with Image.open(image_path) as img:
            img.verify()

    except Exception as e:
        bad_images.append({
            "image_id": row["image_id"],
            "image_path": image_path,
            "error": str(e)
        })

print("Total image-mask pairs:", len(dataset_df))
print("Corrupted images:", len(bad_images))

if bad_images:
    bad_df = pd.DataFrame(bad_images)
    display(bad_df)
else:
    print("No corrupted images found.")


## Train DeepLabV3+

In [ ]:
model = build_deeplabv3plus()
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=combined_loss,
    metrics=[dice_coef, iou_coef],
)

ckpt_path = RESULT_DIR / f"{MODEL_NAME}_best.weights.h5"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(ckpt_path), monitor="val_dice_coef", mode="max",
        save_best_only=True, save_weights_only=True
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.load_weights(str(ckpt_path))
print("Training done for", MODEL_NAME)


## Save Model, History CSV, and Dice/IoU Curves

In [ ]:
model.save(RESULT_DIR / f"{MODEL_NAME}.keras")

history_df = pd.DataFrame(history.history)
history_df.to_csv(RESULT_DIR / f"{MODEL_NAME}_history.csv", index=False)

best_dice = max(history.history["val_dice_coef"])
best_iou = max(history.history["val_iou_coef"])

summary_df = pd.DataFrame([{"model": MODEL_NAME, "val_dice": best_dice, "val_iou": best_iou}])
summary_df.to_csv(RESULT_DIR / f"{MODEL_NAME}_summary.csv", index=False)
print(summary_df)

plt.figure(figsize=(6, 4))
plt.plot(history.history["val_dice_coef"], label="val_dice")
plt.plot(history.history["val_iou_coef"], label="val_iou")
plt.title(f"{MODEL_NAME} training curves")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.savefig(RESULT_DIR / f"{MODEL_NAME}_curves.png", dpi=150)
plt.show()

print("Saved results to:", RESULT_DIR)


## Generate & Save Segmented Results for This Model

In [ ]:
sample_val = val_df.sample(min(8, len(val_df)), random_state=SEED)

fig, axes = plt.subplots(len(sample_val), 3, figsize=(10, 4 * len(sample_val)))

for row_idx, (_, row) in enumerate(sample_val.iterrows()):
    image = cv2.cvtColor(cv2.imread(row["image_path"]), cv2.COLOR_BGR2RGB)
    resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0

    pred = model.predict(resized[np.newaxis, ...], verbose=0)[0, ..., 0]
    pred_mask = (pred > 0.5).astype(np.uint8) * 255

    segmented = cv2.bitwise_and(
        (resized * 255).astype(np.uint8),
        (resized * 255).astype(np.uint8),
        mask=pred_mask,
    )

    out_path = SEGMENTED_DIR / f"{row['image_id']}_segmented.png"
    cv2.imwrite(str(out_path), cv2.cvtColor(segmented, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(SEGMENTED_DIR / f"{row['image_id']}_pred_mask.png"), pred_mask)

    axes[row_idx, 0].imshow(resized); axes[row_idx, 0].set_title("Image"); axes[row_idx, 0].axis("off")
    axes[row_idx, 1].imshow(pred_mask, cmap="gray"); axes[row_idx, 1].set_title("Predicted Mask"); axes[row_idx, 1].axis("off")
    axes[row_idx, 2].imshow(segmented); axes[row_idx, 2].set_title("Segmented Oral Cavity"); axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.savefig(RESULT_DIR / f"{MODEL_NAME}_segmentation_samples.png", dpi=150)
plt.show()

print("Segmented sample images saved to:", SEGMENTED_DIR)


## Done

In [ ]:
print("Everything for this model is saved under:", RESULT_DIR)
print("Files:")
for p in sorted(RESULT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_DIR))
print()
print("Download the whole 'Segmentation Output' folder (or just this model's *_results folder)")
print("from the Kaggle 'Output' panel on the right, then repeat for the other two models.")
